# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge Exploration with `mlcroissant`
This notebook guides you through loading and exploring a dataset using the `mlcroissant` library, referencing Croissant entities via their `@id` for consistency and reproducibility.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
dataset_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(dataset_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The dataset contains multiple record sets (`cr:RecordSet`), fields (`cr:Field`), and columns (`cr:Column`) each referenced by their unique `@id`.

Below, we list the record sets, their fields, and columns by `@id`.

In [ ]:
# List record sets and their fields by @id
record_sets = list(dataset.metadata.record_sets)

print("Record Sets in the dataset:")
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name', '[no name]')}")

    # List fields in each record set
    if 'fields' in rs and rs['fields']:
        print("    Fields:")
        for fld in rs['fields']:
            print(f"      * @id: {fld['@id']} | name: {fld.get('name', '[no name]')} | dataType: {fld.get('dataType', '[unknown]')}")
    # List columns in each record set
    if 'columns' in rs and rs['columns']:
        print("    Columns:")
        for col in rs['columns']:
            print(f"      * @id: {col['@id']} | name: {col.get('name', '[no name]')} | dataType: {col.get('dataType', '[unknown]')}")

if record_sets:
    # Show some example records using the first record set
    print("\nSample records from the first record set:")
    records_iterator = dataset.records(record_set=record_sets[0]['@id'])
    for i, rec in enumerate(records_iterator):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis.

We use the `@id` of each record set for extraction and reference.

In [ ]:
# Extract data from each record set
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Print column names from first record set
example_rs_id = record_set_ids[0] if record_set_ids else None
if example_rs_id:
    print(f"Columns in record set {example_rs_id}: {dataframes[example_rs_id].columns.tolist()}")
    display(dataframes[example_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section includes operations like removing outliers and grouping data by key attributes, referencing all fields/columns via their `@id`.

In [ ]:
# Choose a record set and fields for EDA
# (For demonstration, pick the first record set and first numeric field)
rs_id = example_rs_id

df = dataframes[rs_id]

# Find a numeric field by @id (dataType commonly 'schema:Float' or 'schema:Integer')
numeric_field_id = None
group_field_id = None
fields = next((rs['fields'] for rs in record_sets if rs['@id'] == rs_id), [])
for f in fields:
    if not numeric_field_id and f.get('dataType') in ['schema:Float', 'schema:Integer']:
        numeric_field_id = f['@id']
    if not group_field_id and f.get('dataType') == 'schema:Text':
        group_field_id = f['@id']

print(f"Selected numeric field for filtering (by @id): {numeric_field_id}")
print(f"Selected grouping field (by @id): {group_field_id}")

# Filtering based on numeric field (if field exists)
if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    
    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
else:
    print("No numeric field found for EDA.")

# Grouping (if group field exists)
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using their `@id`.

Below, we show a histogram of the filtered normalized numeric field and, when possible, a boxplot grouped by the selected group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of normalized numeric field
if numeric_field_id and filtered_df.shape[0] > 0:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True)
    plt.title(f"Distribution of normalized '{numeric_field_id}'")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Count")
    plt.show()

# Boxplot by group field
if group_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=filtered_df, x=group_field_id, y=f"{numeric_field_id}_normalized")
    plt.title(f"Normalized '{numeric_field_id}' by '{group_field_id}'")
    plt.xlabel(group_field_id)
    plt.ylabel(f"{numeric_field_id}_normalized")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to explore and analyze a Croissant schema dataset using the `mlcroissant` library.

**Key Observations:**
- Data was loaded, extracted, and viewed referencing all entities by their `@id`.
- Numeric variables were filtered and normalized for further analysis.
- Grouping and visualizations enabled inspection by categorical attributes.

For deeper analysis, refer to specific field and record set `@id`s found in the overview section above: these ensure future-reproducible workflows with Croissant-compliant datasets.